In [18]:
import pandas as pd

df = pd.read_csv("../data/raw/PPR-ALL.csv", encoding='cp1252')
rppi = pd.read_csv("../data/raw/cso_rppi_index.csv")

C:\Users\Admin\AppData\Local\Temp\ipykernel_1364\3190857743.py:3: DtypeWarning: Columns (8: Property Size Description) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/PPR-ALL.csv", encoding='cp1252')


In [19]:
df = df.rename(columns={"Date of Sale (dd/mm/yyyy)": "date", "Price (€)": "price", "Description of Property": "description"})
df.columns = df.columns.str.lower()

In [20]:
print(df.head())

         date                                         address    county  \
0  01/01/2010          5 Braemor Drive, Churchtown, Co.Dublin    Dublin   
1  03/01/2010  134 Ashewood Walk, Summerhill Lane, Portlaoise     Laois   
2  04/01/2010             1 Meadow Avenue, Dundrum, Dublin 14    Dublin   
3  04/01/2010                         1 The Haven, Mornington     Meath   
4  04/01/2010                   11 Melville Heights, Kilkenny  Kilkenny   

  eircode        price not full market price vat exclusive  \
0     NaN  €343,000.00                    No            No   
1     NaN  €185,000.00                    No           Yes   
2     NaN  €438,500.00                    No            No   
3     NaN  €400,000.00                    No            No   
4     NaN  €160,000.00                    No            No   

                             description  \
0  Second-Hand Dwelling house /Apartment   
1          New Dwelling house /Apartment   
2  Second-Hand Dwelling house /Apartment   


In [21]:
df = df.drop(columns="property size description")

In [22]:
df_fmp = df[df["not full market price"] == "No"]
print(len(df_fmp))


776611


In [24]:
df_fmp = df_fmp.drop(columns="not full market price")
display(df_fmp)

,date,address,county,eircode,price,vat exclusive,description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,Second-Hand Dwelling house /Apartment
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",Yes,New Dwelling house /Apartment
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,Second-Hand Dwelling house /Apartment
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,Second-Hand Dwelling house /Apartment
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,Second-Hand Dwelling house /Apartment
...,...,...,...,...,...,...,...
818246,13/11/2026,"DOORADOYLE RD, DOORADOYLE, LIMERICK",Limerick,V94HN2X,"€410,000.00",No,Second-Hand Dwelling house /Apartment
818247,13/11/2026,"DRUMAVILLE, MALIN, DONEGAL",Donegal,F93TW90,"€305,000.00",No,Second-Hand Dwelling house /Apartment
818248,13/11/2026,"GLAS SHEEDY, 8 CLAR AOIBHINN, CLAREMORRIS",Mayo,F12T928,"€380,000.00",No,Second-Hand Dwelling house /Apartment
818249,13/11/2026,"IONA, 27 NUNS WALK, BALLYPHEHANE",Cork,T12A6Y9,"€247,000.00",No,Second-Hand Dwelling house /Apartment


In [25]:
df_fmp["description"].unique()

<StringArray>
['Second-Hand Dwelling house /Apartment',
         'New Dwelling house /Apartment',
        'Teach/Árasán Cónaithe Atháimhe',
             'Teach/Árasán Cónaithe Nua',
             'Teach/?ras?n C?naithe Nua']
Length: 5, dtype: str

In [ ]:
df_fmp["is_new"] = df_fmp["description"].str.contains("new|nua", case=False)
pd.crosstab(df_fmp["description"], df_fmp["is_new"])

is_new,False,True
description,,
New Dwelling house /Apartment,0,138878
Second-Hand Dwelling house /Apartment,637686,0
Teach/?ras?n C?naithe Nua,0,1
Teach/Árasán Cónaithe Atháimhe,43,0
Teach/Árasán Cónaithe Nua,0,3


In [29]:
df_is_new = df_fmp.drop(columns="description")
display(df_is_new)

,date,address,county,eircode,price,vat exclusive,is_new
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,False
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",Yes,True
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,False
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,False
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,False
...,...,...,...,...,...,...,...
818246,13/11/2026,"DOORADOYLE RD, DOORADOYLE, LIMERICK",Limerick,V94HN2X,"€410,000.00",No,False
818247,13/11/2026,"DRUMAVILLE, MALIN, DONEGAL",Donegal,F93TW90,"€305,000.00",No,False
818248,13/11/2026,"GLAS SHEEDY, 8 CLAR AOIBHINN, CLAREMORRIS",Mayo,F12T928,"€380,000.00",No,False
818249,13/11/2026,"IONA, 27 NUNS WALK, BALLYPHEHANE",Cork,T12A6Y9,"€247,000.00",No,False


In [37]:
counts = df_is_new["eircode"].value_counts()
counts[counts > 2]

eircode
D24W9NN    34
V15KH39    11
A96WV79     8
W23VH52     7
D07F6K5     7
           ..
R51F782     3
D24PD7A     3
X91AT82     3
K34EY04     3
A84D271     3
Name: count, Length: 543, dtype: int64

In [ ]:
df_is_new["eircode"] = df_is_new["eircode"].replace("A123456", pd.NA) #removing placeholder


In [ ]:
df_is_new["price"] = df["price"].str.replace("€", "")  #replacing euro sign

In [ ]:
routing = df["eircode"].str[:3]     #checking if there ane any other placeholders
routing.value_counts().tail(30)

eircode
E94    1
C86    1
N62    1
D91    1
W32    1
E12    1
T51    1
Y31    1
F90    1
Y67    1
T24    1
P33    1
P95    1
T36    1
K93    1
F75    1
W94    1
A14    1
P11    1
W25    1
C39    1
C35    1
N35    1
A21    1
R94    1
P19    1
A11    1
T72    1
D85    1
D51    1
Name: count, dtype: int64